# 手动实现 VMC + SR 计算分子基态能量（修复版）

本 notebook 完全手动实现变分蒙特卡洛（VMC）方法，包括：
1. **正确的梯度计算**（完全按照 NetKet 的方式）
2. **量子几何张量（QGT）计算**
3. **随机重配置（SR）优化**
4. **完整的训练循环**

关键修复：
- 使用 VJP 而不是 value_and_grad（与 NetKet 一致）
- 正确处理复数共轭
- 修复 NetKet 对比代码

## 1. 导入库和设置系统

In [1]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
import optax
from functools import partial
from tqdm import tqdm
import matplotlib.pyplot as plt

print(f"JAX version: {jax.__version__}")
print(f"NetKet version: {nk.__version__}")

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


JAX version: 0.5.3
NetKet version: 3.18


## 2. 设置 H₂ 分子系统

In [2]:
# H₂ 分子定义
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)
print("H₂ HF能量:", mf.e_tot)
# FCI 精确基准
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()

print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

# NetKet 哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

# Hilbert 空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)

print(f"\n希尔伯特空间大小: {hi.n_states}")
print(f"希尔伯特空间维度: {hi.size}")

H₂ HF能量: -0.9414806547077985
H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV

希尔伯特空间大小: 4
希尔伯特空间维度: 4


## 3. 定义神经网络 Ansatz

In [4]:
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

def forward(GraphDef_State, x):
    """NetKet 采样器需要的 forward 函数"""
    log_psi, _ = nnx.call(GraphDef_State)(x)
    return log_psi

## 4. 核心函数：能量和梯度计算（使用 VJP）

In [5]:
@partial(jax.jit, static_argnames=("model_forward", "graphdef"))
def compute_energy_and_gradient_vjp(graphdef, state, model_forward, hamiltonian, samples):
    """
    使用 VJP 计算能量和梯度（完全按照 NetKet 的方式）
    
    这与 NetKet 的 expect_forces.py 中的实现完全一致
    """
    n_samples = samples.shape[0]
    
    # 定义 logpsi 函数
    def logpsi_fn(s):
        return model_forward((graphdef, s), samples)
    
    # 计算局部能量
    log_psi = logpsi_fn(state)
    eta, H_sigmaeta = hamiltonian.get_conn_padded(samples)
    logpsi_eta = model_forward((graphdef, state), eta)
    
    log_psi_expanded = jnp.expand_dims(log_psi, axis=-1)
    Eloc = jnp.sum(H_sigmaeta * jnp.exp(logpsi_eta - log_psi_expanded), axis=-1)
    energy = jnp.mean(Eloc)
    
    # 中心化局部能量
    Eloc_centered = Eloc - energy
    
    # 使用 VJP 计算梯度
    # 注意：NetKet 使用 conjugate=True
    _, vjp_fn = jax.vjp(logpsi_fn, state)
    
    # 计算梯度：grad = vjp(conj(Eloc_centered) / n_samples)
    grad = vjp_fn(jnp.conjugate(Eloc_centered) / n_samples)[0]
    
    # 乘以 2（与 NetKet 的 force_to_grad 一致）
    grad = jax.tree_map(lambda g: 2 * g, grad)
    
    return energy, grad

## 5. 核心函数：量子几何张量（QGT）计算

In [6]:
@partial(jax.jit, static_argnames=("model_forward", "graphdef"))
def compute_qgt(model_forward, graphdef, state, samples):
    """
    计算量子几何张量（QGT）
    
    QGT 定义：S_{ij} = cov(O_i*, O_j)
    
    使用中心化的梯度计算
    """
    n_samples = samples.shape[0]
    
    # 定义 logpsi 函数
    def logpsi_single_param(s, x):
        return model_forward((graphdef, s), x)
    
    # 对每个样本计算梯度
    def grad_logpsi_single(x):
        return jax.grad(logpsi_single_param, argnums=0, holomorphic=True)(state, x)
    
    # 批量计算
    grads_tree = jax.vmap(grad_logpsi_single)(samples)
    
    # 展平梯度
    grads_flat, tree_def = jax.tree_util.tree_flatten(grads_tree)
    grads_concat = jnp.concatenate([g.reshape((n_samples, -1)) for g in grads_flat], axis=-1)
    
    # 计算均值和中心化
    mean_grad = jnp.mean(grads_concat, axis=0, keepdims=True)
    centered_grads = grads_concat - mean_grad
    
    # 计算 QGT: S = (1/N) * O^† O
    # 注意：使用中心化的梯度
    qgt = (1.0 / n_samples) * jnp.einsum('si,sj->ij', centered_grads.conj(), centered_grads)
    
    return qgt

## 6. 核心函数：SR 自然梯度计算

In [7]:
from jax import flatten_util

@partial(jax.jit, static_argnames=("model_forward", "graphdef"))
def compute_natural_gradient(model_forward, graphdef, state, samples, energy_grad, epsilon=1e-4):
    """
    计算 SR 自然梯度
    
    自然梯度：Δθ = S^{-1} * g
    """
    # 1. 计算 QGT
    qgt = compute_qgt(model_forward, graphdef, state, samples)
    
    # 2. 正则化 QGT
    n_params = qgt.shape[0]
    qgt_reg = qgt + epsilon * jnp.eye(n_params)
    
    # 3. 展平能量梯度
    g_flat, unflatten_fn = flatten_util.ravel_pytree(energy_grad)
    
    # 4. 解线性方程：S * Δθ = g
    nat_g_flat = jnp.linalg.solve(qgt_reg, g_flat)
    
    # 5. 恢复梯度树结构
    nat_grad = unflatten_fn(nat_g_flat)
    
    return nat_grad, qgt

## 7. 完整训练循环

In [12]:
def train_vmc_sr(
    hamiltonian,
    hilbert,
    model,
    n_iterations=300,
    n_samples=1000,
    learning_rate=0.1,
    sr_epsilon=1e-4,
    use_sr=True,
    seed=21
):
    """
    完整的 VMC + SR 训练循环
    """
    # 初始化模型
    graphdef, model_state = nnx.split(model)
    GraphDef_State = (graphdef, model_state)
    
    # 设置采样器
    edges = [(0, 1), (2, 3)]
    g = nk.graph.Graph(edges=edges)
    single_rule = nk.sampler.rules.FermionHopRule(hilbert=hilbert, graph=g)
    sampler = nk.sampler.MetropolisSampler(
        hilbert, 
        rule=single_rule, 
        n_chains=16, 
        sweep_size=32
    )
    
    # 初始化采样器状态
    sampler_state = sampler.init_state(forward, GraphDef_State, seed=seed)
    
    # 设置优化器
    optimizer = optax.sgd(learning_rate=learning_rate)
    opt_state = optimizer.init(model_state)
    
    # 记录训练过程
    energy_history = []
    
    print(f"\n{'='*70}")
    print(f"开始训练: {'使用 SR' if use_sr else '不使用 SR'}")
    print(f"迭代次数: {n_iterations}, 样本数: {n_samples}, 学习率: {learning_rate}")
    print(f"{'='*70}\n")
    
    for i in tqdm(range(n_iterations), desc="训练进度"):
        # 1. 采样
        sampler_state = sampler.reset(forward, GraphDef_State, state=sampler_state)
        samples, sampler_state = sampler.sample(forward, GraphDef_State, state=sampler_state, chain_length=n_samples // 16)
        samples = samples.reshape(-1, samples.shape[-1])
        
        # 2. 计算能量和梯度
        energy, grads = compute_energy_and_gradient_vjp(
            graphdef, model_state, forward, hamiltonian, samples
        )
        
        # 3. SR 自然梯度（如果启用）
        if use_sr:
            nat_grad, qgt = compute_natural_gradient(
                forward, graphdef, model_state, samples, grads, epsilon=sr_epsilon
            )
            update_grad = nat_grad
        else:
            update_grad = grads
        
        # 4. 参数更新
        updates, opt_state = optimizer.update(update_grad, opt_state, model_state)
        model_state = optax.apply_updates(model_state, updates)
        GraphDef_State = (graphdef, model_state)
        
        # 5. 记录
        energy_history.append(energy.real)
        
        # 6. 打印进度
        if i % 50 == 0:
            error = abs(energy.real - E_fcis[0])
            print(f"\nIter {i:3d} | Energy: {energy.real:.8f} Ha | Error: {error:.6f} Ha")
            if use_sr:
                print(f"         | QGT trace: {jnp.trace(qgt):.4f} | QGT condition: {jnp.linalg.cond(qgt):.2e}")
    
    print(f"\n{'='*70}")
    print(f"训练完成！最终能量: {energy_history[-1]:.8f} Ha")
    print(f"FCI 基准: {E_fcis[0]:.8f} Ha")
    print(f"误差: {abs(energy_history[-1] - E_fcis[0]):.6e} Ha")
    print(f"{'='*70}\n")
    
    return energy_history, model_state

## 8. 运行训练：使用 SR

In [13]:
# 初始化模型
rngs = nnx.Rngs(21)
model_sr = SingleStateAnsatz(n_spin_orbitals=4, hidden_dim=12, rngs=rngs)

# 训练（使用 SR）
energy_history_sr, final_state_sr = train_vmc_sr(
    hamiltonian=ha,
    hilbert=hi,
    model=model_sr,
    n_iterations=500,
    n_samples=1000,
    learning_rate=0.1,
    sr_epsilon=1e-2,
    use_sr=True,
    seed=21
)


开始训练: 使用 SR
迭代次数: 500, 样本数: 1000, 学习率: 0.1



训练进度:   0%|          | 0/500 [00:00<?, ?it/s]

训练进度:   1%|          | 3/500 [00:00<00:18, 26.65it/s]


Iter   0 | Energy: -0.48420472 Ha | Error: 0.531264 Ha
         | QGT trace: 9.3112+0.0000j | QGT condition: 3.78e+34


训练进度:  11%|█         | 55/500 [00:02<00:21, 20.88it/s]

...


Iter 400 | Energy: -0.65240585 Ha | Error: 0.363062 Ha
         | QGT trace: 0.0000+0.0000j | QGT condition: 1.32e+86


训练进度:  91%|█████████ | 456/500 [00:21<00:02, 20.69it/s]


Iter 450 | Energy: -0.65240585 Ha | Error: 0.363062 Ha
         | QGT trace: 0.0000+0.0000j | QGT condition: 2.85e+68


训练进度: 100%|██████████| 500/500 [00:24<00:00, 20.72it/s]


训练完成！最终能量: -0.65240585 Ha
FCI 基准: -1.01546825 Ha
误差: 3.630624e-01 Ha



## 9. 运行训练：不使用 SR（对比）

In [ ]:
# 初始化模型（相同种子）
rngs = nnx.Rngs(21)
model_no_sr = SingleStateAnsatz(n_spin_orbitals=4, hidden_dim=12, rngs=rngs)

# 训练（不使用 SR）
energy_history_no_sr, final_state_no_sr = train_vmc_sr(
    hamiltonian=ha,
    hilbert=hi,
    model=model_no_sr,
    n_iterations=300,
    n_samples=1000,
    learning_rate=0.1,
    sr_epsilon=1e-4,
    use_sr=False,
    seed=21
)

## 10. NetKet 原生实现对比（修复版）

In [11]:
# NetKet 原生实现
rngs = nnx.Rngs(21)
model_netket = SingleStateAnsatz(n_spin_orbitals=4, hidden_dim=12, rngs=rngs)

# 设置采样器
edges = [(0, 1), (2, 3)]
g = nk.graph.Graph(edges=edges)
single_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=g)
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=16, sweep_size=32)

# 创建 MCState
vstate = nk.vqs.MCState(
    sampler=sampler,
    model=model_netket,
    n_samples=1000,
    n_discard_per_chain=10
)

# 设置 SR 和优化器
preconditioner = nk.optimizer.SR(diag_shift=1e-3, holomorphic=True)
optimizer = nk.optimizer.Sgd(0.1)

# 创建 VMC 驱动
vmc = nk.driver.VMC(ha, optimizer, variational_state=vstate, preconditioner=preconditioner)

print(f"\n{'='*70}")
print("NetKet 原生实现训练")
print(f"{'='*70}\n")

# 使用 NetKet 的 run 方法
vmc.run(300, out=None)  # out=None 表示不保存到文件

# 获取最终能量
energy_final_netket = vstate.expect(ha)

print(f"\n{'='*70}")
print(f"NetKet 训练完成！最终能量: {energy_final_netket.mean.real:.8f} Ha")
print(f"{'='*70}\n")

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/netket/vqs/mc/mc_state/state.py:300: UserWarning: n_samples=1000 (1000 per device/MPI rank) does not divide n_chains=16, increased to 1008 (1008 per device/MPI rank)
  self.n_samples = n_samples



NetKet 原生实现训练



100%|██████████| 300/300 [00:14<00:00, 20.61it/s, Energy=-1.015e+00-1.115e-09j ± 2.006e-10 [σ²=4.054e-17, R̂=1.3191]]



NetKet 训练完成！最终能量: -1.01546825 Ha



## 11. 可视化对比结果

In [ ]:
# 绘制能量收敛曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plt.rcParams['font.sans-serif'] = ['PingFang HK']

# 1. 能量收敛曲线
axes[0].plot(energy_history_sr, label='手动实现 (SR)', linewidth=2, color='blue')
axes[0].plot(energy_history_no_sr, label='手动实现 (无 SR)', linewidth=2, color='orange')
axes[0].axhline(y=E_fcis[0], color='red', linestyle=':', label='FCI 基准', linewidth=2)
axes[0].axhline(y=energy_final_netket.mean.real, color='green', linestyle='--', label='NetKet 原生', linewidth=2)
axes[0].set_xlabel('迭代次数', fontsize=12)
axes[0].set_ylabel('能量 (Ha)', fontsize=12)
axes[0].set_title('能量收敛曲线', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# 2. 能量误差（对数坐标）
error_sr = [abs(e - E_fcis[0]) for e in energy_history_sr]
error_no_sr = [abs(e - E_fcis[0]) for e in energy_history_no_sr]

axes[1].semilogy(error_sr, label='手动实现 (SR)', linewidth=2, color='blue')
axes[1].semilogy(error_no_sr, label='手动实现 (无 SR)', linewidth=2, color='orange')
axes[1].axhline(y=abs(energy_final_netket.mean.real - E_fcis[0]), color='green', linestyle='--', label='NetKet 原生', linewidth=2)
axes[1].set_xlabel('迭代次数', fontsize=12)
axes[1].set_ylabel('能量误差 (Ha)', fontsize=12)
axes[1].set_title('能量误差（对数坐标）', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('vmc_sr_comparison_fixed.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. 详细结果分析

In [ ]:
print("\n" + "="*70)
print("详细结果分析")
print("="*70)
print()

print("1. 最终能量对比:")
print(f"   - 手动实现 (SR):    {energy_history_sr[-1]:.8f} Ha")
print(f"   - 手动实现 (无 SR): {energy_history_no_sr[-1]:.8f} Ha")
print(f"   - NetKet 原生:      {energy_final_netket.mean.real:.8f} Ha")
print(f"   - FCI 基准:         {E_fcis[0]:.8f} Ha")
print()

print("2. 能量误差:")
print(f"   - 手动实现 (SR):    {abs(energy_history_sr[-1] - E_fcis[0]):.6e} Ha")
print(f"   - 手动实现 (无 SR): {abs(energy_history_no_sr[-1] - E_fcis[0]):.6e} Ha")
print(f"   - NetKet 原生:      {abs(energy_final_netket.mean.real - E_fcis[0]):.6e} Ha")
print()

print("3. 手动实现与 NetKet 的差异:")
diff_sr_netket = abs(energy_history_sr[-1] - energy_final_netket.mean.real)
print(f"   - SR 实现与 NetKet 的能量差异: {diff_sr_netket:.6e} Ha")
print()

print("="*70)

## 13. 验证梯度一致性

In [ ]:
# 使用最终状态验证梯度计算
print("\n" + "="*70)
print("验证梯度计算的一致性")
print("="*70)

# 准备参数
graphdef, state = nnx.split(model_sr)
nnx_parameters = nnx.State(jax.tree_map(nnx.Param, vstate.parameters))
model_test = nnx.merge(graphdef, nnx_parameters)
graphdef, state = nnx.split(model_test)

# 获取样本
samples = vstate.samples.reshape(-1, 4)

# 手动实现的梯度
energy_manual, grad_manual = compute_energy_and_gradient_vjp(
    graphdef, state, forward, ha, samples
)

# NetKet 的梯度
energy_netket, grad_netket = vstate.expect_and_grad(ha)

print(f"\n能量对比:")
print(f"  手动实现: {energy_manual:.8f} Ha")
print(f"  NetKet:   {energy_netket.mean:.8f} Ha")
print(f"  差异:     {abs(energy_manual - energy_netket.mean):.6e} Ha")

# 比较梯度
def compare_gradients(grad1, grad2):
    max_diff = 0.0
    for key in grad1.keys():
        if isinstance(grad1[key], dict):
            for subkey in grad1[key].keys():
                g1 = grad1[key][subkey]
                g2 = grad2[key][subkey]
                diff = jnp.max(jnp.abs(g1 - g2))
                max_diff = max(max_diff, diff)
    return max_diff

max_grad_diff = compare_gradients(grad_manual, grad_netket)

print(f"\n梯度对比:")
print(f"  最大差异: {max_grad_diff:.6e}")

if max_grad_diff < 1e-5:
    print("  ✓ 梯度计算一致！")
else:
    print("  ✗ 梯度计算存在差异")

print("="*70)

## 14. 总结

In [ ]:
print("\n" + "="*70)
print("总结")
print("="*70)
print()
print("本 notebook 完全手动实现了 VMC + SR 方法，关键改进：")
print()
print("1. 使用 VJP 而不是 value_and_grad:")
print("   - 与 NetKet 的实现方式完全一致")
print("   - 正确处理复数共轭")
print()
print("2. 关键修复点:")
print("   ✓ 中心化局部能量: E_loc_centered = E_loc - <E_loc>")
print("   ✓ 使用共轭: jnp.conjugate(E_loc_centered)")
print("   ✓ 梯度缩放: grad = 2 * grad")
print("   ✓ 使用 VJP: jax.vjp(...)")
print()
print("3. 数学原理:")
print("   - 能量梯度: ∂E/∂θ = 2 Re[cov(O_k, E_loc)]")
print("   - QGT: S_{ij} = cov(O_i*, O_j)")
print("   - 自然梯度: Δθ = S^{-1} * g")
print()
print("="*70)